# Faz 7 — Ablasyon: Loss / Sampler Konfigürasyonları (SwinTransformer)

4 konfigürasyonu **tek fold** üzerinde eğitip test setinde değerlendirir.
İstatistiksel karşılaştırma için 5-fold versiyonu → **Faz7b_CrossVal_Cls.ipynb**

| # | Konfigürasyon | Loss | Sampler | ThreshTuning |
|---|---|---|---|---|
| 1 | Baseline | BCE | Hayır | Hayır |
| 2 | FocalBCE | Focal | Hayır | Hayır |
| 3 | FocalBCE+Sampler | Focal | Evet | Hayır |
| 4 | FocalBCE+Sampler+Thresh | Focal | Evet | Evet |

**Çıktı:** `outputs/results/ablation_results.json` → Faz6 ve Faz8 bu dosyayı kullanır.

---
## 0. Ortam

In [ ]:
import os, sys, json, time
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = not IS_KAGGLE and os.path.exists('/content')
IS_LOCAL  = not IS_KAGGLE and not IS_COLAB

env_name = 'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'

if IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working/abdomen1')
    WORK_DIR     = Path('/kaggle/working')
    DATA_DIR     = Path('/kaggle/input') / 'abdomen-acikveri'
elif IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path('/content/drive/MyDrive/abdomen')
    WORK_DIR     = Path('/content')
    DATA_DIR     = Path('/content/kaggle_data')
else:
    PROJECT_ROOT = Path('/Users/ramazanpolat/Desktop/datasets/abdomen')
    WORK_DIR     = PROJECT_ROOT / 'outputs'
    DATA_DIR     = PROJECT_ROOT

OUT_DIR     = WORK_DIR / 'outputs'
SPLIT_DIR   = OUT_DIR / 'splits'
CKPT_DIR    = OUT_DIR / 'checkpoints' / 'ablation'
RESULTS_DIR = OUT_DIR / 'results'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault('ABDOMEN_SPLIT_DIR', str(SPLIT_DIR))
os.environ.setdefault('ABDOMEN_TRAIN_DIR', str(DATA_DIR / 'Egitim Verisi'))

FOLD = 0
SUPER_CLASSES = [
    'acute_cholecystitis', 'kidney_ureter_stone', 'acute_pancreatitis',
    'aortic_aneurysm_dissection', 'acute_appendicitis', 'acute_diverticulitis',
]

print(f'Ortam      : {env_name}')
print(f'CKPT_DIR   : {CKPT_DIR}')
print(f'RESULTS_DIR: {RESULTS_DIR}')

In [ ]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

---
## 1. Kütüphaneler ve src

In [ ]:
import subprocess
if IS_KAGGLE or IS_COLAB:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'timm', 'pydicom', 'opencv-python-headless', 'pandas', 'numpy', 'tqdm', 'scikit-learn'],
        check=True
    )

# Kaggle / Colab: repo klon
GITHUB_URL = 'https://github.com/ramazan2020/abdomen1.git'
if not IS_LOCAL:
    if not (PROJECT_ROOT / '.git').exists():
        subprocess.run(['git', 'clone', '--depth=1', GITHUB_URL, str(PROJECT_ROOT)], check=True)
    else:
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], capture_output=True)
    print(f'Repo: {PROJECT_ROOT}')

from src.config   import ClsConfig
from src.splits   import load_fold
from src.train_cls import build_model, train_one_fold, tune_thresholds
print('src modülleri ✓')

---
## 2. Ablasyon Konfigürasyonları

In [ ]:
from dataclasses import dataclass
from typing import Optional

ABLATION_CONFIGS = [
    {
        'key':                  'baseline',
        'name':                 'Baseline (BCE)',
        'use_focal':            False,
        'use_weighted_sampler': False,
        'tune_thresholds':      False,
    },
    {
        'key':                  'focalbce',
        'name':                 'FocalBCE',
        'use_focal':            True,
        'use_weighted_sampler': False,
        'tune_thresholds':      False,
    },
    {
        'key':                  'focalbce_sampler',
        'name':                 'FocalBCE+Sampler',
        'use_focal':            True,
        'use_weighted_sampler': True,
        'tune_thresholds':      False,
    },
    {
        'key':                  'focalbce_sampler_thresh',
        'name':                 'FocalBCE+Sampler+Thresh',
        'use_focal':            True,
        'use_weighted_sampler': True,
        'tune_thresholds':      True,
    },
]

BASE_CFG = dict(
    backbone      = 'swin_base_patch4_window12_384.ms_in22k_ft_in1k',
    num_classes   = 6,
    input_size    = 384,
    batch_size    = 16,
    epochs        = 50,
    lr            = 2e-4,
    weight_decay  = 1e-4,
    warmup_epochs = 3,
    focal_gamma   = 2.0,
    accum_steps   = 2,
)

def ckpt_path(cfg_key: str) -> Path:
    return CKPT_DIR / f'{cfg_key}_fold{FOLD}_best.pth'

def thresh_path(cfg_key: str) -> Path:
    return CKPT_DIR / f'{cfg_key}_fold{FOLD}_thresholds.json'

print(f'{len(ABLATION_CONFIGS)} konfigürasyon, Fold {FOLD}')
for c in ABLATION_CONFIGS:
    ckpt_ok = '✓' if ckpt_path(c['key']).exists() else '—'
    print(f'  [{ckpt_ok}] {c["name"]}')

---
## 3. Eğitim

Checkpoint mevcutsa atlanır.

In [ ]:
for cfg in ABLATION_CONFIGS:
    cp = ckpt_path(cfg['key'])
    if cp.exists():
        print(f'[ATLA] {cfg["name"]} — checkpoint mevcut: {cp.name}')
        continue

    print(f'\n═══ Eğitim: {cfg["name"]} ═══')
    train_cfg = ClsConfig(
        **BASE_CFG,
        use_focal            = cfg['use_focal'],
        use_weighted_sampler = cfg['use_weighted_sampler'],
        fold                 = FOLD,
        ckpt_dir             = str(CKPT_DIR),
    )
    best = train_one_fold(fold=FOLD, cfg=train_cfg)

    # train_one_fold'un kaydettiği best.pth'yi cfg key ile yeniden adlandır
    default_ckpt = CKPT_DIR / f'cls_fold{FOLD}' / 'best.pth'
    if default_ckpt.exists():
        default_ckpt.rename(cp)
        print(f'Checkpoint → {cp.name}')

    # Threshold tuning
    if cfg['tune_thresholds']:
        model = build_model(train_cfg).to(DEVICE)
        ck    = torch.load(str(cp), map_location=DEVICE)
        model.load_state_dict(ck.get('model', ck))
        val_cases = load_fold(FOLD, 'val')
        thresholds = tune_thresholds(model, val_cases, SUPER_CLASSES, DEVICE)
        thresh_path(cfg['key']).write_text(json.dumps(thresholds, indent=2))
        print(f'Thresholds → {thresh_path(cfg["key"]).name}')
        del model
        if DEVICE == 'cuda': torch.cuda.empty_cache()

    print(f'  mAP={best["mAP"]:.4f}  macroF1={best["macroF1"]:.4f}')

print('\nEğitim tamamlandı ✓')

---
## 4. Test Seti Değerlendirmesi

In [ ]:
import torchvision.transforms as T
from src.datasets import SliceMultiLabelDataset

# Test split yükle
_test_csv = SPLIT_DIR / 'holdout_test.csv'
if not _test_csv.exists():
    _test_csv = SPLIT_DIR / f'fold{FOLD}_test.csv'
if not _test_csv.exists():
    _splits_csv = SPLIT_DIR / 'splits.csv'
    if _splits_csv.exists():
        _df_all  = pd.read_csv(_splits_csv)
        df_test  = _df_all[_df_all['split'] == 'test']
        print(f'Test seti splits.csv[split==test]: {len(df_test)} satır')
    else:
        df_test = None
        print('[UYARI] Test CSV bulunamadı — val seti kullanılacak')
else:
    df_test = pd.read_csv(_test_csv)
    print(f'Test seti: {len(df_test)} satır ({_test_csv.name})')


def evaluate_config(cfg: dict) -> Optional[dict]:
    cp = ckpt_path(cfg['key'])
    if not cp.exists():
        print(f'  [ATLA] {cfg["name"]}: checkpoint yok')
        return None

    # Model yükle
    train_cfg = ClsConfig(**BASE_CFG, fold=FOLD)
    model = build_model(train_cfg).to(DEVICE)
    ck    = torch.load(str(cp), map_location=DEVICE)
    model.load_state_dict(ck.get('model', ck.get('model_state_dict', ck)))
    model.eval()

    # Threshold
    tp = thresh_path(cfg['key'])
    thresholds = json.loads(tp.read_text()) if tp.exists() else {c: 0.5 for c in SUPER_CLASSES}

    # DataLoader
    _eval_cases = df_test['case'].tolist() if df_test is not None else load_fold(FOLD, 'val')
    transform = T.Compose([
        T.Resize((BASE_CFG['input_size'], BASE_CFG['input_size'])),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    dataset = SliceMultiLabelDataset(_eval_cases, input_size=BASE_CFG['input_size'])
    loader  = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=(DEVICE=='cuda'))

    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=cfg['name'], leave=False):
            imgs   = batch['image'].to(DEVICE)
            probs  = torch.sigmoid(model(imgs)).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(batch['labels'].numpy())

    probs  = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)

    per_class_f1 = {}
    for i, cls in enumerate(SUPER_CLASSES):
        tau     = thresholds.get(cls, 0.5)
        pred_b  = (probs[:, i] >= tau).astype(int)
        tp_v = int(((pred_b == 1) & (labels[:, i] == 1)).sum())
        fp_v = int(((pred_b == 1) & (labels[:, i] == 0)).sum())
        fn_v = int(((pred_b == 0) & (labels[:, i] == 1)).sum())
        prec = tp_v / max(tp_v + fp_v, 1)
        rec  = tp_v / max(tp_v + fn_v, 1)
        per_class_f1[cls] = round(2 * prec * rec / max(prec + rec, 1e-9), 4)

    top5 = round(float(np.mean(list(per_class_f1.values()))), 4)

    del model
    if DEVICE == 'cuda': torch.cuda.empty_cache()

    return {
        'key':            cfg['key'],
        'name':           cfg['name'],
        'top5_mean_f1':   top5,
        'per_class_f1':   per_class_f1,
        'thresholds_used': thresholds,
    }


ablation_results = []
for cfg in ABLATION_CONFIGS:
    r = evaluate_config(cfg)
    if r:
        ablation_results.append(r)
        print(f'  {cfg["name"]}: Top-5 Mean F1 = {r["top5_mean_f1"]:.4f}')

print(f'\n{len(ablation_results)}/{len(ABLATION_CONFIGS)} konfigürasyon değerlendirildi')

---
## 5. Sonuçları Kaydet ve Özet

In [ ]:
out = {'fold': FOLD, 'configs': ablation_results}
(RESULTS_DIR / 'ablation_results.json').write_text(json.dumps(out, indent=2, ensure_ascii=False))
print(f'Kaydedildi: {RESULTS_DIR}/ablation_results.json')

print()
print('=' * 52)
print(f'  Ablasyon Özeti — Fold {FOLD}')
print('=' * 52)
for r in ablation_results:
    print(f'  {r["name"]:<35} {r["top5_mean_f1"]:.4f}')
print('=' * 52)
print()
print('→ Faz6_Harici_Test.ipynb: model karşılaştırması')
print('→ Faz7b_CrossVal_Cls.ipynb: 5-fold CV (istatistiksel doğrulama)')
print('→ Faz8_Makale_Analiz.ipynb: ablation_results.json kullanılır')

---
## 6. Hızlı Grafik

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if not ablation_results:
    print('Sonuç yok — önce eğitim + değerlendirme hücrelerini çalıştırın.')
else:
    names  = [r['name'] for r in ablation_results]
    scores = [r['top5_mean_f1'] for r in ablation_results]
    palette = ['#9E9E9E', '#4C72B0', '#55A868', '#C44E52'][:len(names)]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(f'Ablasyon — Fold {FOLD}', fontsize=12, fontweight='bold')

    # Bar chart
    axes[0].bar(range(len(names)), scores, color=palette, edgecolor='black',
                linewidth=0.5, alpha=0.88)
    axes[0].set_xticks(range(len(names)))
    axes[0].set_xticklabels(names, rotation=20, ha='right', fontsize=9)
    axes[0].set_ylabel('Top-5 Mean F1')
    axes[0].set_ylim(0, 1)
    axes[0].set_title('Konfigürasyon Karşılaştırması')
    for i, s in enumerate(scores):
        axes[0].text(i, s + 0.01, f'{s:.3f}', ha='center', va='bottom', fontsize=9)

    # Per-class F1
    x = range(len(SUPER_CLASSES))
    for i, r in enumerate(ablation_results):
        vals = [r['per_class_f1'].get(c, 0) for c in SUPER_CLASSES]
        axes[1].plot(x, vals, 'o-', color=palette[i], label=r['name'], linewidth=1.8)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([c[:12] for c in SUPER_CLASSES], rotation=25, ha='right', fontsize=8)
    axes[1].set_ylabel('F1')
    axes[1].set_ylim(0, 1)
    axes[1].legend(fontsize=8)
    axes[1].set_title('Sınıf Bazında F1')
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    fig.savefig(RESULTS_DIR / 'ablation_plot.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Grafik kaydedildi: ablation_plot.png')